In [1]:
import pandas as pd
import numpy as np

In [61]:
# =============================================================================
# BASE PATHS — Update these to match your local directory structure
# =============================================================================
your_path_here = '/storage/Arushi/090526_EvoAge/kg_formation/data_collection/'

# NCBI gene info for C. elegans
NCBI_GENE_INFO_PATH = f'{your_path_here}databases_for_mapping/ncbi/Caenorhabditis_elegans.gene_info'
WB_gene_locus = f'{your_path_here}databases_for_mapping/wormbase/wormbase.WS240.gene_ids.txt'


# Pre-processed WID/FICE gene-gene pickle
FIC_PATH = f'{your_path_here}fic/network2.csv'

# Final output path
OUTPUT_PATH = '/storage/Arushi/090526_EvoAge/kg_formation/processed_data/fic/fic_CELE_GENE_GENE.csv'

In [50]:
ncbi_cele_gene

,#tax_id,GeneID,Symbol,LocusTag,Synonyms,dbXrefs,chromosome,map_location,description,type_of_gene,Symbol_from_nomenclature_authority,Full_name_from_nomenclature_authority,Nomenclature_status,Other_designations,Modification_date,Feature_type
0,6239,171590,homt-1,Y74C9A.3,-,WormBase:WBGene00022277|AllianceGenome:WB:WBGe...,I,-,Alpha N-terminal protein methyltransferase 1,protein-coding,homt-1,Alpha N-terminal protein methyltransferase 1,O,Alpha N-terminal protein methyltransferase 1,20250308,-
1,6239,171591,nlp-40,Y74C9A.2,-,WormBase:WBGene00022276|AllianceGenome:WB:WBGe...,I,-,Peptide P4,protein-coding,nlp-40,Peptide P4,O,Peptide P4,20250308,-
2,6239,171592,rcor-1,Y74C9A.4,-,WormBase:WBGene00022278|AllianceGenome:WB:WBGe...,I,-,REST corepressor rcor-1,protein-coding,rcor-1,REST corepressor rcor-1,O,REST corepressor rcor-1,20250308,-
3,6239,171593,sesn-1,Y74C9A.5,-,WormBase:WBGene00022279|AllianceGenome:WB:WBGe...,I,-,Sestrin homolog,protein-coding,sesn-1,Sestrin homolog,O,Sestrin homolog,20250308,-
4,6239,171594,pgs-1,Y48G1C.4,-,WormBase:WBGene00021677|AllianceGenome:WB:WBGe...,I,-,CDP-diacylglycerol--glycerol-3-phosphate 3-pho...,protein-coding,pgs-1,CDP-diacylglycerol--glycerol-3-phosphate 3-pho...,O,CDP-diacylglycerol--glycerol-3-phosphate 3-pho...,20250308,-
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
46922,6239,80510347,azyx-1,F42G4.11,-,WormBase:WBGene00306133|AllianceGenome:WB:WBGe...,II,-,Alternative N-terminal open reading frame of Z...,protein-coding,azyx-1,Alternative N-terminal open reading frame of Z...,O,Alternative N-terminal open reading frame of Z...,20250308,-
46923,6239,80510348,F54D10.10,F54D10.10,-,WormBase:WBGene00306132|AllianceGenome:WB:WBGe...,II,-,Uncharacterized protein,protein-coding,F54D10.10,Uncharacterized protein,O,Uncharacterized protein,20241028,-
46924,6239,80510349,F58B6.11,F58B6.11,-,WormBase:WBGene00306125|AllianceGenome:WB:WBGe...,III,-,Uncharacterized protein,protein-coding,F58B6.11,Uncharacterized protein,O,Uncharacterized protein,20241028,-
46925,6239,80510350,Y34B4A.20,Y34B4A.20,-,WormBase:WBGene00306131|AllianceGenome:WB:WBGe...,X,-,Uncharacterized protein,protein-coding,Y34B4A.20,Uncharacterized protein,O,Uncharacterized protein,20241028,-


In [56]:
ncbi_cele_gene[ncbi_cele_gene['LocusTag'] == 'C38C6.2']

,#tax_id,GeneID,Symbol,LocusTag,Synonyms,dbXrefs,chromosome,map_location,description,type_of_gene,Symbol_from_nomenclature_authority,Full_name_from_nomenclature_authority,Nomenclature_status,Other_designations,Modification_date,Feature_type
3248,6239,175125,atgp-2,C38C6.2,-,WormBase:WBGene00000225|AllianceGenome:WB:WBGe...,II,-,Aamy domain-containing protein;Glycosyl hydrol...,protein-coding,atgp-2,Aamy domain-containing protein;Glycosyl hydrol...,O,Aamy domain-containing protein;Glycosyl hydrol...,20250308,-


In [43]:
# =============================================================================
# Load NCBI gene info and build dictionaries
# =============================================================================

ncbi_cele_gene = pd.read_csv(NCBI_GENE_INFO_PATH, sep='\t')


# Strip 'CELE_' prefix from LocusTag
ncbi_cele_gene['LocusTag'] = ncbi_cele_gene['LocusTag'].str.replace('CELE_', '', regex=False)

NCBI_Cele_gene_Locus_2_GeneSymd_ict = dict(zip(ncbi_cele_gene['Symbol'], ncbi_cele_gene['LocusTag']))
NCBI_Cele_gene_Locus_2_GeneSymd_ict
# Dictionary: LocusTag → Description (used for head/tail_detail_name)
ncbi_locustag_to_description = dict(zip(ncbi_cele_gene['LocusTag'], ncbi_cele_gene['description']))

print(f"NCBI genes loaded: {len(ncbi_cele_gene):,}")
print(f"LocusTag → Description dictionary size: {len(ncbi_locustag_to_description):,}")

NCBI genes loaded: 46,927
LocusTag → Description dictionary size: 46,927


In [31]:
wormbase_gene_logus = pd.read_csv(f'{your_path_here}wormbase/wormbase.WS240.gene_ids.txt', sep = '\t')
wormbase_gene_logus_dict =  dict(zip(wormbase_gene_logus[' GeneID '], wormbase_gene_logus[' (Locus name) ']))


In [57]:
C_Ele_G_G_FIC = pd.read_csv(f'{FIC_PATH}')
C_Ele_G_G_FIC_Exp = C_Ele_G_G_FIC[C_Ele_G_G_FIC['type'] == 'experiment']
C_Ele_G_G_FIC_Exp

C_Ele_G_G_FIC_Exp['Head'] = np.nan
C_Ele_G_G_FIC_Exp
# Define a function to map the values
def map_head(gene_name):
    if gene_name in wormbase_gene_logus_dict:
        return wormbase_gene_logus_dict[gene_name]
    elif gene_name in wormbase_gene_logus_dict:
        return wormbase_gene_logus_dict[gene_name]
    else:
        return None

# Apply the function to fill the 'HEAD' column
C_Ele_G_G_FIC_Exp['Head'] = C_Ele_G_G_FIC_Exp['gene1'].apply(map_head)

C_Ele_G_G_FIC_Exp['Tail'] = np.nan

# Define a function to map the values
def map_head(gene_name):
    if gene_name in wormbase_gene_logus_dict:
        return wormbase_gene_logus_dict[gene_name]
    elif gene_name in wormbase_gene_logus_dict:
        return wormbase_gene_logus_dict[gene_name]
    else:
        return None

# Apply the function to fill the 'HEAD' column
C_Ele_G_G_FIC_Exp['Tail'] = C_Ele_G_G_FIC_Exp['gene2'].apply(map_head)

C_Ele_G_G_FIC_Exp[C_Ele_G_G_FIC_Exp['Head'].isna()]
C_Ele_G_G_FIC_Exp[C_Ele_G_G_FIC_Exp['Tail'].isna()]

def remove_last_alpha(text):
    text = str(text)
    
    # remove last character only if it is an alphabet
    if text[-1].isalpha():
        return text[:-1]
    
    return text

C_Ele_G_G_FIC_Exp['Head'] = C_Ele_G_G_FIC_Exp['Head'].apply(remove_last_alpha)
C_Ele_G_G_FIC_Exp['Tail'] = C_Ele_G_G_FIC_Exp['Tail'].apply(remove_last_alpha)
C_Ele_G_G_FIC_Exp

/tmp/ipykernel_1638179/3192405211.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  C_Ele_G_G_FIC_Exp['Head'] = np.nan
/tmp/ipykernel_1638179/3192405211.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  C_Ele_G_G_FIC_Exp['Head'] = C_Ele_G_G_FIC_Exp['gene1'].apply(map_head)
/tmp/ipykernel_1638179/3192405211.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https

,gene1,gene2,type,biological_process_value,molecular_function_value,cellular_component_value,integrated_value,INTERDOM_value,RNAseq_value,Microarray_value,phyprofile_tanimato_value,phyprofile_pearson_value,phyprofile_mutual_value,score,Head,Tail
70,WBGene00000002,WBGene00000225,experiment,0.000000,0.085344,0.003354,0.000000,0.000000,0.419671,0.233105,0.888889,0.869143,0.000000,0.0,F27C8.1,C38C6.2
77,WBGene00000004,WBGene00000225,experiment,0.000000,0.085344,0.003354,0.000000,0.000000,0.530789,0.220729,1.000000,0.790116,1.000000,0.0,F52H2.2,C38C6.2
78,WBGene00000007,WBGene00006438,experiment,0.000000,0.000000,0.000000,-0.148963,0.000000,0.517043,0.441895,0.666667,0.038482,0.000000,0.0,T11F9.4,C01F6.6
80,WBGene00000013,WBGene00006593,experiment,0.011892,0.000000,0.000000,0.000000,0.000000,0.117578,-0.054779,0.000000,0.000000,0.000000,0.0,C50F2.10,C07F11.1
90,WBGene00000018,WBGene00000417,experiment,0.065631,0.000000,0.000000,0.538517,0.000708,0.509631,-0.030221,0.666667,0.954213,0.000000,0.0,M79.1,C48D1.2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
108318,WBGene00021634,WBGene00023421,experiment,0.000000,0.000000,0.000000,0.952596,0.000000,0.754547,0.467885,0.571429,0.478091,0.258294,0.0,Y47G6A.4,B0564.11
108402,WBGene00021910,WBGene00021910,experiment,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,Y55B1BM.1,Y55B1BM.1
108407,WBGene00021914,WBGene00022866,experiment,0.000000,0.000000,0.000000,0.752746,0.000000,0.000000,0.334527,0.777778,0.349310,0.000000,0.0,Y55B1BR.4,ZK1240.1
108462,WBGene00022149,WBGene00023498,experiment,0.000000,0.000000,0.000526,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,Y71G12B.9,ZK678.1


In [59]:
# Drop the specified columns
columns_to_drop = ['gene1','gene2','type','biological_process_value','molecular_function_value','cellular_component_value','integrated_value','INTERDOM_value','RNAseq_value','Microarray_value','phyprofile_tanimato_value','phyprofile_pearson_value','phyprofile_mutual_value','score']
C_Ele_G_G_FIC_Exp.drop(columns=columns_to_drop, inplace=True)

C_Ele_G_G_FIC_Exp['Relation'] = 'Gene_Gene'
C_Ele_G_G_FIC_Exp['Relation_Source'] = 'FICE'
C_Ele_G_G_FIC_Exp['head_type'] = 'Gene'
C_Ele_G_G_FIC_Exp['tail_type'] = 'Gene'

# Reorder the columns in the DataFrame
desired_columns = ['Head', 'Relation', 'Tail', 'head_type', 'tail_type', 'Relation_Source']
C_Ele_G_G_FIC_Exp = C_Ele_G_G_FIC_Exp[desired_columns]

C_Ele_G_G_FIC_Exp['relation_type'] = np.nan
C_Ele_G_G_FIC_Exp['kg_source'] = C_Ele_G_G_FIC_Exp['Relation_Source']
C_Ele_G_G_FIC_Exp['head_detail_name'] = C_Ele_G_G_FIC_Exp['Head'].map(ncbi_locustag_to_description)
C_Ele_G_G_FIC_Exp['tail_detail_name'] = C_Ele_G_G_FIC_Exp['Tail'].map(ncbi_locustag_to_description)
C_Ele_G_G_FIC_Exp['head_id_is'] = 'NCBI_ID'
C_Ele_G_G_FIC_Exp['tail_id_is'] = 'NCBI_ID'
C_Ele_G_G_FIC_Exp['species'] = 'C.elegans'
C_Ele_G_G_FIC_Exp
C_Ele_G_G_FIC_Exp
C_Ele_G_G_FIC_Exp = C_Ele_G_G_FIC_Exp[~C_Ele_G_G_FIC_Exp['head_detail_name'].isna()]
C_Ele_G_G_FIC_Exp = C_Ele_G_G_FIC_Exp[~C_Ele_G_G_FIC_Exp['tail_detail_name'].isna()]
C_Ele_G_G_FIC_Exp
C_Ele_G_G_FIC_Exp

/tmp/ipykernel_1638179/1606207200.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  C_Ele_G_G_FIC_Exp.drop(columns=columns_to_drop, inplace=True)
/tmp/ipykernel_1638179/1606207200.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  C_Ele_G_G_FIC_Exp['Relation'] = 'Gene_Gene'
/tmp/ipykernel_1638179/1606207200.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing

,Head,Relation,Tail,head_type,tail_type,Relation_Source,relation_type,kg_source,head_detail_name,tail_detail_name,head_id_is,tail_id_is,species
70,F27C8.1,Gene_Gene,C38C6.2,Gene,Gene,FICE,NaN,FICE,Large neutral amino acids transporter small su...,Aamy domain-containing protein;Glycosyl hydrol...,NCBI_ID,NCBI_ID,C.elegans
77,F52H2.2,Gene_Gene,C38C6.2,Gene,Gene,FICE,NaN,FICE,Large neutral amino acids transporter small su...,Aamy domain-containing protein;Glycosyl hydrol...,NCBI_ID,NCBI_ID,C.elegans
78,T11F9.4,Gene_Gene,C01F6.6,Gene,Gene,FICE,NaN,FICE,Amino acid transporter protein 6,Na(+)/H(+) exchange regulatory cofactor-like p...,NCBI_ID,NCBI_ID,C.elegans
80,C50F2.10,Gene_Gene,C07F11.1,Gene,Gene,FICE,NaN,FICE,Antibacterial factor-related peptide 2,TIR domain-containing protein,NCBI_ID,NCBI_ID,C.elegans
90,M79.1,Gene_Gene,C48D1.2,Gene,Gene,FICE,NaN,FICE,F-actin binding domain-containing protein;Tyro...,Cell death protein 3 subunit p17,NCBI_ID,NCBI_ID,C.elegans
...,...,...,...,...,...,...,...,...,...,...,...,...,...
108318,Y47G6A.4,Gene_Gene,B0564.11,Gene,Gene,FICE,NaN,FICE,RNA interference defective protein 10,RNA interference defective protein 11,NCBI_ID,NCBI_ID,C.elegans
108402,Y55B1BM.1,Gene_Gene,Y55B1BM.1,Gene,Gene,FICE,NaN,FICE,Stromal interaction molecule 1,Stromal interaction molecule 1,NCBI_ID,NCBI_ID,C.elegans
108407,Y55B1BR.4,Gene_Gene,ZK1240.1,Gene,Gene,FICE,NaN,FICE,MAGUK family,RING-type domain-containing protein,NCBI_ID,NCBI_ID,C.elegans
108462,Y71G12B.9,Gene_Gene,ZK678.1,Gene,Gene,FICE,NaN,FICE,ML domain-containing protein;RING-type domain-...,Protein lin-15A,NCBI_ID,NCBI_ID,C.elegans


In [63]:
C_Ele_G_G_FIC_Exp.to_csv(f'{OUTPUT_PATH}',index = None)
OUTPUT_PATH

'/storage/Arushi/090526_EvoAge/kg_formation/processed_data/fic/fic_CELE_GENE_GENE.csv'